# Books to Scrape: Web Scraping with Python

This beginner project uses **Requests** and **Beautiful Soup** to collect book titles, prices, ratings, and product-page links from the practice website [Books to Scrape](https://books.toscrape.com/), then saves the results to a CSV file.

> Run the notebook manually when you want to refresh the data. It does not run an endless loop or schedule itself.

## 1. Import libraries

Install the dependencies listed in `requirements.txt` before running the notebook.

In [ ]:
import csv
import time
from datetime import date
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
OUTPUT_FILE = Path("BooksToScrape.csv")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; beginner-learning-project/1.0)"
}


## 2. Define a scraper

The function below visits each catalogue page, extracts the book information shown in the listing, and follows the **next** link until there are no more pages. A short pause between pages helps avoid sending requests too quickly.

In [ ]:
def scrape_books(max_pages=None, delay_seconds=0.5):
    """Scrape book listings from Books to Scrape.

    Args:
        max_pages: Optional page limit for a quick test. Use None for all pages.
        delay_seconds: Pause between page requests.

    Returns:
        A list of dictionaries containing book details.
    """
    books = []
    page_url = BASE_URL
    pages_visited = 0

    with requests.Session() as session:
        session.headers.update(HEADERS)

        while page_url and (max_pages is None or pages_visited < max_pages):
            response = session.get(page_url, timeout=20)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            book_cards = soup.select("article.product_pod")
            if not book_cards:
                raise ValueError(f"No book listings found on {page_url}")

            for card in book_cards:
                title_element = card.select_one("h3 a")
                price_element = card.select_one(".price_color")
                rating_element = card.select_one("p.star-rating")

                if title_element is None or price_element is None:
                    continue

                rating_classes = rating_element.get("class", []) if rating_element else []
                rating = next(
                    (value for value in rating_classes if value != "star-rating"),
                    "Unknown"
                )

                books.append({
                    "Title": title_element.get("title", title_element.get_text(strip=True)),
                    "Price": price_element.get_text(strip=True).replace("£", ""),
                    "Rating": rating,
                    "Product URL": urljoin(page_url, title_element.get("href", "")),
                    "Scraped Date": date.today().isoformat(),
                })

            pages_visited += 1
            next_link = soup.select_one("li.next a")
            page_url = urljoin(page_url, next_link["href"]) if next_link else None

            if page_url and delay_seconds > 0:
                time.sleep(delay_seconds)

    return books


## 3. Run the scraper

For a quick test, set `max_pages=1`. To scrape all catalogue pages, leave `max_pages=None`.

In [ ]:
books = scrape_books(max_pages=1)  # Change to None to scrape the full catalogue.
print(f"Collected {len(books)} books.")
books[:5]


## 4. Save the results to CSV

The CSV is saved in the same folder as this notebook, so the project does not depend on a user-specific Mac file path.

In [ ]:
if not books:
    raise ValueError("No books were collected. Check the website connection and scraper selectors.")

df = pd.DataFrame(books)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"Saved {len(df)} rows to: {OUTPUT_FILE.resolve()}")

df.head()


## 5. Inspect the collected data

Use pandas to check the shape, column types, missing values, and a few summary statistics.

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
print("\nPrice summary:")
display(df["Price"].describe())


## Notes and limitations

- This is a learning exercise using the site's intentionally scrape-friendly catalogue.
- The `max_pages=1` setting is a quick test; use `max_pages=None` to collect all catalogue pages.
- Prices are stored as numeric-looking values without the currency symbol; the source website displays prices in pounds sterling (£).
- Ratings are recorded as labels such as `One`, `Two`, and `Five`, matching the website's star-rating classes.
- The output file is overwritten each time the notebook runs. Make a copy first if you want to preserve an earlier scrape.
- The notebook makes requests only when you run the scraping cell; there is no background schedule.